# Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
spark_path = "E:\\Spark\\spark-3.2.0-bin-hadoop2.7"

In [ ]:
import findspark
findspark.init(spark_path)

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [ ]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

In [ ]:
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "10g")
    .config("spark.sql.files.maxPartitionBytes", "268435456")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

In [ ]:
spark

<h1> Topics </h1>

1. Reading Files (parquet)
2. Narrow Operations
   - `filter`
   - `withColumn`: adding/modifying a column
   - `select`: selecting relevant columns
3. Wide Operations
   - Joins
     - Sort-Merge Join
     - Broadcast Join
   - GroupBy
     - `count` & `sum`
     - `countDistinct`

# Reading File

In [ ]:
transactions_file = "./data/transactions.parquet"
df_transactions = spark.read.parquet(transactions_file)

In [ ]:
df_transactions.rdd.getNumPartitions()

In [ ]:
df_transactions.show(5, False)

In [ ]:
customers_file = "./data/customers.parquet"
df_customers = spark.read.parquet(customers_file)

In [ ]:
df_customers.show(5, False)

# Narrow Transformations
- `filter` rows where `city='boston'`
- `add` a new column: adding `first_name` and `last_name`
- `alter` an exisitng column: adding 5 to `age` column
- `select` relevant columns

In [ ]:
df_narrow_transform = (
    df_customers
    .filter(F.col("city") == "boston")
    .withColumn("first_name", F.split("name", " ").getItem(0))
    .withColumn("last_name", F.split("name", " ").getItem(1))
    .withColumn("age", F.col("age") + F.lit(5))
    .select("cust_id", "first_name", "last_name", "age", "gender", "birthday")
)

df_narrow_transform.write.format("noop").mode("overwrite").save("./data/test/df_narrow_transform.parquet")

In [ ]:
df_narrow_transform.show(7, False)

In [ ]:
df_customer_gt_50 = (
    df_customers
    .filter(F.col("age").cast("int") > 50)
)
df_customer_gt_50.write.format("noop").mode("overwrite").save("./data/test/df_customer_gt_50.parquet")

# Wide Transformations
1. Joins
   - Sort Merge Join
   - Broadcast Join
2. GroupBy
   - `count`
   - `countDistinct`
   - `sum`

## 1. Joins

### Sort Merge Join

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [ ]:
df_joined = (
    df_transactions.join(
        df_customers,
        how="inner",
        on="cust_id"
    )
)

In [ ]:
df_joined.write.format("noop").mode("overwrite").save("./data/test/df_joined.parquet")

In [ ]:
df_joined.explain(True)

### Broadcast Join

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)    #100MB

In [ ]:
df_broadcast_joined = (
    df_transactions.join(
        F.broadcast(df_customers),
        how="inner",
        on="cust_id"
    )
)

In [ ]:
df_broadcast_joined.write.format("noop").mode("overwrite").save("./data/test/df_broadcast_joined.parquet")

## 2. GroupBy

### GroupBy Count

In [ ]:
df_city_counts = (
    df_transactions
    .groupBy("city")
    .count()
)

In [ ]:
df_city_counts.show(5, False)

In [ ]:
df_txn_amt_city = (
    df_transactions
    .groupBy("city")
    .agg(F.sum("amt").alias("txn_amt"))
)

In [ ]:
df_txn_amt_city.show(5, False)

### GroupBy Count Distinct 

In [ ]:
df_txn_per_city = (
    df_transactions
    .groupBy("city")
    .agg(F.countDistinct("txn_id").alias("txn_count"))
)

In [ ]:
df_txn_per_city.show(5, False)

In [ ]:
spark.stop()